# King County House Prices: the refactored notebook

This is the **refactored copy** of `King-County.ipynb`. The original notebook is unchanged.

**What changed**

| Before (original notebook) | Now |
|---|---|
| Data cleaning written cell by cell | Functions in `cleaning.py` |
| Feature engineering written cell by cell (with loops) | Functions in `features.py` |
| Everything run by hand, in order | One reusable pipeline in `pipeline.py` |
| No tests | Tests in `tests/` (run them with `uv run --no-sync pytest`) |

The **modeling section stays in the notebook**, as the project asks. Only the definition of `X` and `y` now comes from the pipeline.

The exploratory data analysis (histograms, box plots, maps) is not repeated here. You find it in the original notebook.

## Importing the libraries

In [1]:
import os

import pandas as pd
import plotly.express as px
import skops.io as sio
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Our own code: the refactored data cleaning, feature engineering and pipeline.
from cleaning import clean_data, load_data
from features import add_features
from pipeline import run_pipeline, split_features_and_target

## Data cleaning and feature engineering

The steps of the original notebook are now three function calls:

1. `load_data()` reads the CSV file.
2. `clean_data()` removes the 33-bedroom house, rebuilds `sqft_basement`, fills `view` and `waterfront`, and creates `last_known_change`.
3. `add_features()` adds `sqft_price`, `center_distance` and `water_distance`.

In [2]:
raw_data = load_data()
cleaned_data = clean_data(raw_data)
kc_data = add_features(cleaned_data)

print("Raw data:     ", raw_data.shape)
print("Cleaned data: ", cleaned_data.shape)
print("With features:", kc_data.shape)

Raw data:      (21597, 21)
Cleaned data:  (21596, 20)
With features: (21596, 23)


The pipeline does the same in **one call**. We check that both ways give the same table.

In [3]:
kc_data_from_pipeline = run_pipeline()
print("Same result as the step-by-step version:", kc_data_from_pipeline.equals(kc_data))

Same result as the step-by-step version: True


A quick check of the results: no missing values are left, and the new columns have sensible values.

In [4]:
print("Missing values left:", kc_data.isnull().sum().sum())
kc_data[["last_known_change", "sqft_price", "center_distance", "water_distance"]].describe().round(2)

Missing values left: 0


,last_known_change,sqft_price,center_distance,water_distance
count,21596.00,21596.00,21596.00,21596.00
mean,1972.95,59.73,17.53,5.54
std,28.95,45.62,10.15,5.93
min,1900.00,0.16,0.34,0.00
25%,1954.00,28.14,10.33,1.91
50%,1977.00,46.88,14.46,3.42
75%,1999.00,78.02,22.54,6.36
max,2015.00,399.48,70.19,57.43


# Modeling

This section is **unchanged** from the original notebook. Only the next cell is new: `split_features_and_target()` from the pipeline replaces the four cells that defined `drop_lst`, `all_features`, `X` and `y`.

In [5]:
# `X` holds the model input, `y` is the price (the target).
# The pipeline removes `price`, `sqft_price` (it is built from the price) and `date`.
X, y = split_features_and_target(kc_data)

In [6]:
# Split the data into training and test sets. We reserve 30% for evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [7]:
# Inspect the size of each split.
print("X_train (features for the model to learn from): ", X_train.shape)
print("y_train (labels for the model to learn from): ", y_train.shape)
print("X_test (features to test the model's accuracy against): ", X_test.shape)
print("y_test (labels to test the model's accuracy with): ", y_test.shape)

X_train (features for the model to learn from):  (15117, 20)
y_train (labels for the model to learn from):  (15117,)
X_test (features to test the model's accuracy against):  (6479, 20)
y_test (labels to test the model's accuracy with):  (6479,)


In [8]:
# The training data is now shuffled rather than sorted by index.
X_train.head()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,zipcode,lat,long,sqft_living15,sqft_lot15,last_known_change,center_distance,water_distance
3372,1770000490,2,1.75,1060,16470,1.0,0.0,0.0,3,7,1060,0,98072,47.7409,-122.089,1790,16748,1977,17.039745,10.681894
753,8682300890,2,2.50,2380,6600,1.0,0.0,0.0,3,8,2380,0,98053,47.7170,-122.020,1870,6600,2010,19.391272,9.342139
1418,8073000550,4,3.75,3190,17186,2.0,1.0,4.0,3,10,3190,0,98178,47.5115,-122.246,2290,13496,1999,12.943085,0.000000
8178,7212680850,3,2.50,1730,6930,2.0,0.0,0.0,3,8,1730,0,98003,47.2621,-122.308,1780,6930,1994,41.002661,9.194998
2254,8880600070,4,2.00,1870,8750,1.0,0.0,2.0,3,7,1870,0,98022,47.1985,-122.001,1770,8207,1977,51.087178,29.623377


After the split, we can start modeling.
We continue with scikit-learn, where the overall workflow is usually the same:

- import the algorithm
- instantiate the model with its hyperparameters
- choose the feature columns
- fit the model with `.fit(X_train, y_train)`
- evaluate it on the test data

This repeated pattern makes it easy to compare several models in a structured way.



In [9]:
# Instantiate the model.
model_lin_reg = LinearRegression()

Next, we decide which variables to pass to the model.
We begin with a very simple baseline that uses only one predictor: the house `grade`.



In [10]:
# Select the variables passed to the model.
variables = ["grade"]

In [11]:
# Train the model.
model_lin_reg.fit(X_train[variables], y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[211503.07]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['grade']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-1.078e+06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(1)


In [12]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test[variables], y_test))
        * (X_test.shape[0] - 1)
        / (X_test.shape[0] - len(variables) - 1),
        2,
    ),
)

Adj. R^2: 0.43


**R²** measures the proportion of variance in the target variable that is explained by the model.
**Adjusted R²** is a related metric that also accounts for the number of predictors, which makes it more useful when comparing models of different complexity.

Interpretation:
- `1.0` means the model explains all observed variance
- `0.0` means the model explains none of it

In our case, using only `grade` explains about **43%** of the variance in the price on the test set.
That is a useful start, but it suggests that additional variables may improve the model.

Let us inspect the available features again.



In [13]:
# Show the names of the variables in the dataset.
X_train.columns

Index(['id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'zipcode', 'lat', 'long', 'sqft_living15',
       'sqft_lot15', 'last_known_change', 'center_distance', 'water_distance'],
      dtype='str')

We can use more features to predict the price.
The age of the house may also matter, so next we try a linear regression that includes both `grade` and age-related information.



In [14]:
# Instantiate the model.
model_lin_reg = LinearRegression()

In [15]:
# Select the variables passed to the model.
variables = ["grade", "last_known_change"]

In [16]:
# Train the model.
model_lin_reg.fit(X_train[variables], y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[250617.39, -3456.69]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['grade','last_known_change']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,5.442e+06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


In [17]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test[variables], y_test))
        * (X_test.shape[0] - 1)
        / (X_test.shape[0] - len(variables) - 1),
        2,
    ),
)

Adj. R^2: 0.48


With this additional variable, the model explains about **48%** of the variance in the price.



So far, we have only modeled linear relationships between the predictors and price.
However, some effects may be curved rather than straight.

One simple way to capture that is to add squared terms. Instead of fitting only:

$price = b \cdot grade + c$

we can also fit:

$price = a \cdot grade^{2} + b \cdot grade + c$

This is another form of feature engineering. We will apply it to the dataset and see whether the richer representation improves performance.



In [18]:
# Create only second-order polynomial features (^2).
poly = PolynomialFeatures(2)

In [19]:
# Create copies of the training and test data.
X_train_poly = X_train.copy()
X_test_poly = X_test.copy()

# Drop the `id` column.
X_train_poly = X_train_poly.drop(columns=["id"])
X_test_poly = X_test_poly.drop(columns=["id"])

In [20]:
# Create transformed variables with `poly`.
X_train_sq = poly.fit_transform(X_train_poly)

# Apply the same transformation to the test data.
X_test_sq = poly.transform(X_test_poly)

In [21]:
# Instantiate the model.
model_lin_reg = LinearRegression()

In [22]:
# Train the model with the squared features.
model_lin_reg.fit(X_train_sq, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](210,)","[ 0.,-0., 0.,..., 0., 0., 0.]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.664e+07
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,210
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(37)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](210,)","[4.26e+12,1.32e+12,3.54e+11,...,4.42e-04,4.41e-04,4.25e-04]"


In [23]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test_sq, y_test))
        * (X_test_sq.shape[0] - 1)
        / (X_test_sq.shape[0] - X_test_sq.shape[1] - 1),
        2,
    ),
)

Adj. R^2: 0.75


Adding squared features improved the model clearly: the adjusted R² rose from 0.48 to 0.75.



Adjusted R² gives us one useful summary of model quality, but it is also important to inspect the errors directly.
A **residual analysis** can reveal systematic problems that a single score might hide.  

For easier interpretation, we calculate the percentage difference between the predicted and true prices.



In [24]:
# Error analysis
# To inspect the model errors more closely, create a new DataFrame with the
# columns `price` (the true value), `lat`, and `long`.
y_predictions = model_lin_reg.predict(X_test_sq)
df_error = pd.DataFrame(y_test)
df_error["latitude"] = X_test["lat"]
df_error["longitude"] = X_test["long"]
df_error["id"] = X_test["id"]
df_error.head(2)

,price,latitude,longitude,id
3686,132500.0,47.5030,-122.356,6303401050
10247,415000.0,47.6966,-122.324,1995200200


In [25]:
# Reset the index before adding the predicted price as a new column.
df_error.reset_index(inplace=True, drop=True)
df_error.head(2)

<notebook cell>:2: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_error.reset_index(inplace=True, drop=True)


,price,latitude,longitude,id
0,132500.0,47.5030,-122.356,6303401050
1,415000.0,47.6966,-122.324,1995200200


In [26]:
# Add the predicted price and calculate the difference.
df_error["price_prediction"] = y_predictions.round(2)
df_error["price_difference"] = (df_error["price_prediction"] - df_error["price"]).round(
    2
)
df_error["price_difference_percent"] = (
    (df_error["price_difference"] / df_error["price"]) * 100
).round(2)
df_error.head(2)

,price,latitude,longitude,id,price_prediction,price_difference,price_difference_percent
0,132500.0,47.5030,-122.356,6303401050,183826.08,51326.08,38.74
1,415000.0,47.6966,-122.324,1995200200,382829.44,-32170.56,-7.75


In [27]:
fig = px.scatter_map(
    df_error,
    lat="latitude",
    lon="longitude",
    hover_data=["price", "price_prediction", "id"],
    color="price_difference_percent",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7,
    height=400,
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

We can already see a few strong outliers in the residual plot, so it is worth inspecting the largest one more closely.



In [28]:
df_error[
    df_error["price_difference_percent"] == df_error["price_difference_percent"].max()
]

,price,latitude,longitude,id,price_prediction,price_difference,price_difference_percent
2125,169317.0,47.6248,-122.301,5016003230,716736.32,547419.32,323.31


In [29]:
X_test[X_test["id"] == 9272202260]

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,zipcode,lat,long,sqft_living15,sqft_lot15,last_known_change,center_distance,water_distance
18318,9272202260,3,1.0,1200,7000,2.0,0.0,0.0,1,7,1200,0,98116,47.5883,-122.384,3290,6000,1908,11.527542,3.49912


Let us inspect these houses in more detail.
King County provides a helpful property lookup tool on [this page](https://localscape.property/#kingcountyassessor/My-Property), where you can search by parcel ID.

To inspect the outlier:
- change the selector in the top-left corner from `Address` to `Parcel ID`
- enter the `id` of the outlier

Under **Basic Property Characteristics** and in the **Aerial Images** view, we can see that there is no longer a house on the property. That explains why our model estimated a much higher sale value than the actual transaction suggests.



## Regularization and Hyperparameter Tuning

After adding squared features, the model has many more inputs.
Some of them may contribute useful signal, while others mainly capture noise.
If a model learns noise instead of general patterns, it becomes **overfitted**.

**Regularization** helps control this problem by shrinking coefficients and, in some cases, pushing some of them all the way to zero.
That allows the model to keep the more informative variables while reducing reliance on weaker ones.

Here we use [ElasticNet](https://scikit-learn.org/stable/modules/linear_model.html#elastic-net), which combines L1 and L2 regularization.

Polynomial features have very different scales. A [Pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline) applies polynomial expansion and [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) inside each training fold to prevent validation-data leakage and help convergence. Supply the original feature columns without the identifier, not the already expanded matrix.

The compact grid keeps both penalties active; pure L2 regularization is better handled by Ridge. Five-fold validation compares 45 fits using training data only. The held-out test set is reserved for final evaluation.
A [TransformedTargetRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html) also standardizes training prices within each fold, making the regularization grid meaningful on a unit-scale target. Predictions are automatically converted back to dollars.

To find a good amount of regularization, we test multiple hyperparameter values and compare the results with [GridSearch](https://scikit-learn.org/stable/modules/grid_search.html#grid-search) and [cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation).



In [30]:
# Specify the hyperparameters to tune.
# `alpha` controls the amount of regularization.
param_grid = {
    "model__regressor__alpha": [0.01, 0.1, 1],
    "model__regressor__l1_ratio": [0.2, 0.5, 0.8],
}

# Keep preprocessing with the estimator for validation and inference.
elastic_pipeline = Pipeline(
    [
        ("polynomial", PolynomialFeatures(2, include_bias=False)),
        ("scaler", StandardScaler()),
        (
            "model",
            TransformedTargetRegressor(
                regressor=ElasticNet(max_iter=5000, tol=1e-4, precompute=True),
                transformer=StandardScaler(),
            ),
        ),
    ]
)

# Pass the model to a parameter grid with 5-fold cross-validation.
elastic_cv = GridSearchCV(
    elastic_pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    verbose=2,
    n_jobs=1,
    error_score="raise",
)

# Train and optimize the model with grid search.
elastic_cv.fit(X_train_poly, y_train)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.2; total time=   0.2s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.2; total time=   0.3s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.2; total time=   0.4s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.2; total time=   0.6s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.2; total time=   0.3s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.5; total time=   0.3s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.5; total time=   0.2s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.5; total time=   0.2s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.5; total time=   0.9s
[CV] END model__regressor__alpha=0.01, model__regressor__l1_ratio=0.5; total time=   1.0s
[CV] END model__regressor__alpha=0.01, m

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...rdScaler()))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__regressor__alpha': [0.01, 0.1, ...], 'model__regressor__l1_ratio': [0.2, 0.5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric 

In [31]:
# Display the best parameters found by the grid search.
elastic_cv.best_params_

{'model__regressor__alpha': 0.01, 'model__regressor__l1_ratio': 0.2}

In [32]:
# GridSearchCV already refits the winning pipeline on all training rows.
elastic = elastic_cv.best_estimator_

In [33]:
# Evaluate how well the model performs on the test data.
print("Test R^2:", round(elastic.score(X_test_poly, y_test), 3))

Test R^2: 0.844


Compare held-out R^2 with the earlier models; regularization does not guarantee improvement. Counting nonzero coefficients is not an exact degrees-of-freedom correction for ElasticNet, so we report ordinary R^2.  

As discussed above, regularization can set some regression coefficients to zero, effectively removing less useful variables from the model.  

The following code displays part of the learned coefficient vector. Check whether any coefficients became zero rather than assuming they did.



In [34]:
# Coefficients are in standardized feature and target units, not dollars.
elastic.named_steps["model"].regressor_.coef_[0:5]

array([-0.        , -0.        ,  0.        ,  0.00116475, -0.        ])

In [35]:
# Error analysis
# To inspect the model errors more closely, create a new DataFrame with the
# columns `price` (the true value), `lat`, and `long`.
y_predictions = elastic.predict(X_test_poly)
df_error = pd.DataFrame(y_test)
df_error["latitude"] = X_test["lat"]
df_error["longitude"] = X_test["long"]
df_error["id"] = X_test["id"]
df_error.head(2)

,price,latitude,longitude,id
3686,132500.0,47.5030,-122.356,6303401050
10247,415000.0,47.6966,-122.324,1995200200


In [36]:
# Reset the index before adding the predicted price as a new column.
df_error.reset_index(inplace=True, drop=True)
df_error.head(2)

<notebook cell>:2: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_error.reset_index(inplace=True, drop=True)


,price,latitude,longitude,id
0,132500.0,47.5030,-122.356,6303401050
1,415000.0,47.6966,-122.324,1995200200


In [37]:
# Add the predicted price and calculate the difference.
df_error["price_prediction"] = y_predictions.round(2)
df_error["price_difference"] = (df_error["price_prediction"] - df_error["price"]).round(
    2
)
df_error["price_difference_percent"] = (
    (df_error["price_difference"] / df_error["price"]) * 100
).round(2)
df_error.head(2)

,price,latitude,longitude,id,price_prediction,price_difference,price_difference_percent
0,132500.0,47.5030,-122.356,6303401050,221701.18,89201.18,67.32
1,415000.0,47.6966,-122.324,1995200200,411290.09,-3709.91,-0.89


In [38]:
fig = px.scatter_map(
    df_error,
    lat="latitude",
    lon="longitude",
    hover_data=["price", "price_prediction", "id"],
    color="price_difference_percent",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7,
    height=400,
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

In [39]:
df_error[
    df_error["price_difference_percent"] == df_error["price_difference_percent"].max()
]

,price,latitude,longitude,id,price_prediction,price_difference,price_difference_percent
3344,150000.0,47.7112,-121.315,1437500015,599743.09,449743.09,299.83


In [40]:
X_test[X_test["id"] == 5111400086]

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,zipcode,lat,long,sqft_living15,sqft_lot15,last_known_change,center_distance,water_distance
17183,5111400086,3,1.0,1250,53143,1.0,0.0,0.0,5,6,1250,0,98038,47.4235,-122.051,1820,217800,1945,26.872648,14.812078


Let us take a closer look at this house on [this page](https://localscape.property/#kingcountyassessor/My-Property).

This house sold in 2018 for $295,000. King County appraisal data also reports historical values, and under **Historical Value** we can see that the **Total Assessed Value** in 2014 was $212,000.
That means the sale price was still unusually low relative to the broader context, which helps explain why it stands out so strongly in the residual analysis.



## Save the Model

We have now trained a model that can predict house prices in King County.
The final step is to save the trained pipeline with [skops](https://skops.readthedocs.io/en/stable/modules/classes.html#module-skops.io). Polynomial expansion and scaling are included: inference expects the original training feature columns in the same order, without the identifier.

If you want to explore the optional application stretch goal, see [bonus_solution/](./bonus_solution/). It contains a small working FastAPI + Postgres + Docker example that is meant as a reference, not a required solution path.



In [41]:
import os

import skops.io as sio

# Create the directory if it does not already exist.
os.makedirs("model", exist_ok=True)

with open("model/model.bin", "wb") as f_out:
    sio.dump(elastic, f_out)

### Conclusion

Despite the presence of outliers, we built a model that explains roughly **84%** of the variance in the target.  

The notebook also shows an important modeling lesson: **feature engineering, nonlinear terms, and regularization can all make a meaningful difference in predictive performance**.

